# 02 - Train a DQN Model Offline

This notebook shows the offline training workflow in Mouse Core:

Each event is tokenized by `Tokenizer` as text and embedded by the agent's separate
`Embedding` module.

- `type: "text"` — each field has its own `format=` and is tokenized as its own run.
  `format=` is an f-string using payload names (`{observation}`, `{reward:g}`).
  `input_field` optionally filters one name or a list of names. A list matches
  when any selected value is present; missing selected values are `None`.
  Omit the filter to expose all payload fields. Literal formats need no callback.
- `type: "token"` — literal `format` → one token ID. Nonempty output must encode to
  exactly one token. An optional `input_field` gates emission on presence. The
  action-request marker reuses the existing `\n` token without adding vocabulary
  entries.
- `type: "image"` — vision span (VL checkpoints; not used here)

There is no whole-step `format=`. Fields emit in `input_fields` order, each as a
separate HF tokenize call, so BPE never merges across fields. The coordinator's
`_episode_start` event emits `episode_prompt` as ordinary text before the first
observation of each outer episode. All inner episodes share that sequence and can attend
to the prompt. The producer and sampler own episode boundaries; the packer never inserts
a prompt. Incremental decode packs only new events. A new sample clears that stream's
context and calls `cache.reset_rows`. Collection omits zero rewards and false ending
flags. Absent or None input fields emit nothing. Collection records the zero-based
`repeat_episode_index` only on each inner reset event. The tokenizer renders it as a
one-based `episode N` header and an `x` action marker before that reset observation,
then inserts a blank line
after inner termination or truncation. These formatting newlines have no readout.
Missing fields stay absent in decoded views and optional token fields are skipped.
`_terminated` and `_truncated` are payload metadata mapped to the `terminated` and
`truncated` objective columns. `action_requested` emits the existing `\n` token with
`head_output=True`. Passive and terminal steps have no readout. Omit `provider` to match all providers, or select one by name. Input filters
emit when any selected value is present and non-None.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in
`15_inference.ipynb`.


In [ ]:
import json
from copy import deepcopy
from functools import partial

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    Augmenter,
    DataLoader,
    EnvironmentProtocol,
    Tokenizer,
    compose,
    environment_episode_ranges,
    load_stores_from_hub,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    prepare_decode,
    push_model_to_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount

# Repositories
# Hugging Face dataset repo for load_stores_from_hub
DATASET_ID = "mouse-example-frozenlake-dataset"
# Hugging Face model repo for push_model_to_hub
MODEL_ID = "mouse-example-model-offline"

# Environment
# number of discrete actions predicted by the head
MAX_ACTIONS = 4
# vocabulary size for discrete observations
MAX_OBS_DISCRETE = 64
# max steps per episode (matches collect)
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task (matches collect)
REPEAT_COUNT = 20

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
HEAD_LAYERS = 1
# final prediction linear's initialization
HEAD_SCALE = 0.1

# Compute text before each value readout
COMPUTE_TOKENS = 8

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer train cycles (print cadence)
NUM_CYCLES = 2
# optimizer updates per cycle (passed to run_train)
TRAIN_STEPS = 50

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Target interpolation rates
# delayed embedding
POLYAK_TAU_EMBEDDING = 0.1
# delayed backbone
POLYAK_TAU_BACKBONE = 0.1
# delayed value head
POLYAK_TAU_VALUE = 0.001

# Runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Payload names are selected explicitly; the underscore is only a convention.
environment_episode_ranges = partial(
    environment_episode_ranges,
    protocol=EnvironmentProtocol(
        episode_start="_episode_start",
        terminated="_terminated",
        truncated="_truncated",
    ),
)

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved
`Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)

## Data Pipeline

`DataLoader` samples complete outer tasks with
`sample_ranges=environment_episode_ranges`, keeping all repeated lake episodes together.
Each sampled task gets a unique sequence ID so attention and objective transitions
cannot cross between tasks.

Pass exactly one of `token_budget` or `events_budget`. Each batch contains whole samples
up to that budget. An oversized first sample raises. Token budgets count packed tokens,
including episode prompts; event budgets count stored source events, not decision
readouts.

The pipeline is
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`:

| Stage       | Role                                                                                                             |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| `Augmenter` | Permutes actions and observations consistently within a sampled task; action permutations also update `_q_star`. |
| `Tokenizer` | Tokenizes matching fields per event, groups tokens by `step_index`, and retains explicit objective labels.       |

`train_transform = compose(stages=(augmenter.events, tokenizer.encode))` is passed as
the loader’s `transform`. `next_batch()` returns `(inputs, objective_data)` as
TensorDicts; pass `objective_data["step_sequence_ids"]` as the objective’s `group_id`.
[Inference](15_inference.ipynb) uses the saved tokenizer without augmentation so
selected actions match the environment.

For per-event encoding, configure `Tokenizer` with a flat `input_fields` list,
`step_field="step_index"`, and `episode_field="episode_index"`, then pass
`transform=tokenizer.encode` directly to the loader. It accumulates event tokens until
the step marker changes. Use `partial(step_sample_ranges, steps_per_sample=N)` to sample
N whole steps within an episode. Augmentation for this path must transform event
payloads before encoding; `augmenter.events` preserves source events and shares
episode-keyed draws.

`COMPUTE_TOKENS` sets a constant number of compute tokens before every value readout.
Set it to zero to disable compute text. The field filters on `action_requested`
and uses the literal format `" ." * COMPUTE_TOKENS`. No callback or runtime context
is needed. The configured text is saved with the tokenizer and reused at inference.

Qwen encodes each spaced period as one token (ID 659); ten bare periods can merge
into just one token. A setup check verifies the configured count. Passive and
terminal events emit no compute text. Compute tokens count toward the batch budget.

Only the newline has `head_output=True`. The unchanged DQN loss trains the compute
positions through causal attention. Notebook 15 uses a fixed inference count so
rebuilding a cached history reproduces its text.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": 'episode {repeat_episode_index+1}\n',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": '{action_name(action)}',
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ',{observation}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ',r={reward:g}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": " ." * COMPUTE_TOKENS,
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)


if COMPUTE_TOKENS < 0:
    raise ValueError("Compute token count must be nonnegative")

# Check once at setup: each compute position must be one identical token.
compute_token_ids = tokenizer.tokenizers["text"].encode(" .", add_special_tokens=False)
if len(compute_token_ids) != 1:
    raise ValueError("Compute text must encode to exactly one token")
for compute_count in (COMPUTE_TOKENS,):
    if (
        tokenizer.tokenizers["text"].encode(
            " ." * compute_count, add_special_tokens=False
        )
        != compute_token_ids * compute_count
    ):
        raise ValueError("Repeated compute text must preserve the exact token count")

train_transform = compose(stages=(augmenter.events, tokenizer.encode))


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=environment_episode_ranges,
    transform=train_transform,
    prefetch=4,
    num_workers=4,
)

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.

The delayed encoder is a frozen copy. Separate backup value/action heads track the
online Q head(s) with independent Polyak rates. Only online parameters enter the
optimizer.

The embedding, backbone, and heads are constructed in float32. Mouse Core's `AdamW`
requires trainable parameters in float32; choose `dtype=torch.float32` at construction
rather than casting a trainable backbone to bf16. The example uses FlexAttention
for attention in both training and inference.

For mixed-precision compute with fp32 weights, set `train_autocast_dtype=torch.bfloat16`
on `Backbone`; `decode_autocast_dtype` controls cached inference independently.
Configure these on the backbone instead of wrapping its forward call in
`torch.autocast`.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)

q_head = (
    QHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)

## Train

Each cycle runs `TRAIN_STEPS` optimizer updates. One-step DQN uses the delayed action
head to select the next action and the delayed value head to evaluate it.
`temperature=0.0` selects the highest-scoring action. Rewards pass through unchanged,
and `boundary_discount` ends the return at outer task termination or truncation. Inner
episode endings keep the return connected across repeats.

`environment_episode_ranges` selects complete outer tasks from the event log. Unfinished
tails remain in storage. The objective raises if a training target needs a value beyond
its sampled group; zero discount at outer termination or truncation ends the return.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_token_mask`) and `objective_data` (training labels and step mappings).
`objective_data["head_step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    temperature=0.0,
    gate=None,
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        group_id = objective_data["step_sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            backup_action = q_head(delayed_out["z"])
        losses, metrics = objective(
            objective_data=objective_data.to(device),
            group_id=group_id.to(device),
            head_step_ids=objective_data["head_step_ids"],
            predictions=q_head(out["z"]),
            backup_value_predictions=backup_value_head(delayed_out["z"]),
            backup_action_predictions=backup_action,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
    assert loss is not None
    return (loss, metrics)

## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the
checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}"
    )

loader.close()

## Save To The Hub

The checkpoint contains the encoder and separate head(s) in `modules`. The model weights
and tokenizer packing specification share `MODEL_ID`. Reconstruct the same component
structure before loading. Checkpoints from the old architecture with a projection are
not compatible with this example.


In [ ]:
modules.eval().to("cpu")
model_url = push_model_to_hub(
    model=modules,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    private=False,
    clear=True,
)
print(f"Pushed encoder, heads, and tokenizer to {model_url}")